# 02 — Structured Tool Use

**Model:** DeepSeek-V3 via Nebius AI Studio  
**Pattern:** Tool Calling  
**Difficulty:** ⭐⭐⭐☆☆

---

## The Problem

An LLM's knowledge is frozen at training time. For any task requiring current data, calculations, or interaction with external systems, the model must be able to **call tools** — not just reason about them.

Tool use sounds simple but breaks in practice for two reasons:
1. **Error handling** — tools fail. A robust agent must catch errors and decide whether to retry, use a fallback, or give up.
2. **Parallel calls** — when multiple tools are needed, calling them sequentially is slow. Modern models support requesting several tool calls in one pass.

This notebook shows how to handle both with LangGraph's `ToolNode`.

## Architecture

```
┌──────────────────────────────────────────────────────────┐
│                     LangGraph StateGraph                  │
│                                                           │
│   ┌─────────┐       ┌──────────────┐                     │
│   │  Agent  │──────▶│  Tool calls? │                     │
│   │  Node   │       └──────┬───────┘                     │
│   └────▲────┘        yes   │    no → END                 │
│        │                   ▼                             │
│        │           ┌───────────────┐                     │
│        └───────────│  ToolNode     │                     │
│                    │ (executes all │                     │
│                    │  tool calls)  │                     │
│                    └───────────────┘                     │
└──────────────────────────────────────────────────────────┘
```

## When to Use This

- Any agent that needs real-time data (search, APIs, databases)
- Math-heavy tasks where hallucinating numbers is unacceptable
- Actions with side effects (sending emails, writing files)

**Tradeoff:** Tool use adds latency. Keep tools fast (<2s ideally) and fail loudly with descriptive error messages so the agent can recover.

## Setup

In [ ]:
import os
import math
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode

# DeepSeek-V3 — strong instruction following and native parallel tool call support
llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)

print("LLM ready:", llm.model_name)

## Tool Definitions

We define three tools:
- `web_search` — wraps Tavily for live web results
- `calculate` — evaluates a safe arithmetic expression
- `get_unit_conversion` — handles common unit conversions

Each tool uses the `@tool` decorator which automatically extracts the name, description, and argument schema from the function signature and docstring.

In [ ]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def web_search(query: str) -> str:
    """Search the web for current information. Use for facts, news, prices, or anything time-sensitive."""
    try:
        results = tavily.search(query=query, max_results=3)
        snippets = [r["content"] for r in results.get("results", [])]
        if not snippets:
            return "No results found for this query."
        return "\n\n".join(snippets)
    except Exception as e:
        return f"Search failed: {str(e)}"


@tool
def calculate(expression: str) -> str:
    """
    Evaluate a mathematical expression and return the numeric result.
    Supports: +, -, *, /, **, sqrt(), log(), sin(), cos(), pi, e.
    Example: '2 ** 10 + sqrt(144)'
    """
    try:
        # Restrict to safe math namespace only
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        allowed["__builtins__"] = {}
        result = eval(expression, allowed)  # noqa: S307
        return str(result)
    except Exception as e:
        return f"Calculation error: {str(e)}"


@tool
def get_unit_conversion(value: float, from_unit: str, to_unit: str) -> str:
    """
    Convert a value between common units.
    Supported pairs: km/miles, kg/lbs, celsius/fahrenheit, meters/feet.
    """
    conversions = {
        ("km", "miles"): lambda x: x * 0.621371,
        ("miles", "km"): lambda x: x * 1.60934,
        ("kg", "lbs"): lambda x: x * 2.20462,
        ("lbs", "kg"): lambda x: x * 0.453592,
        ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
        ("meters", "feet"): lambda x: x * 3.28084,
        ("feet", "meters"): lambda x: x * 0.3048,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key not in conversions:
        return f"Conversion from {from_unit} to {to_unit} is not supported."
    result = conversions[key](value)
    return f"{value} {from_unit} = {result:.4f} {to_unit}"


tools = [web_search, calculate, get_unit_conversion]
print(f"Registered {len(tools)} tools:", [t.name for t in tools])

## Bind Tools to the LLM

Binding tells the model what tools are available. The model can then output `tool_calls` in its response — a structured list of tool names and arguments — instead of plain text.

In [ ]:
llm_with_tools = llm.bind_tools(tools)

## Graph Nodes

In [ ]:
def agent_node(state: MessagesState) -> dict:
    """The agent decides what to do: respond or call tools."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


def route_after_agent(state: MessagesState) -> str:
    """If the last message has tool calls, execute them. Otherwise, we're done."""
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        print(f"\n[Router] Executing {len(last_msg.tool_calls)} tool call(s): "
              f"{[tc['name'] for tc in last_msg.tool_calls]}")
        return "tools"
    print("\n[Router] No tool calls — returning final answer.")
    return END

## Build the Graph

`ToolNode` is a LangGraph built-in that handles executing all tool calls in the last AIMessage, wrapping each result in a `ToolMessage`, and returning the updated message list. It handles errors gracefully by catching exceptions and returning them as tool output rather than crashing.

In [ ]:
tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.set_entry_point("agent")
builder.add_conditional_edges("agent", route_after_agent)
builder.add_edge("tools", "agent")  # after tools run, agent gets results

graph = builder.compile()
print("Graph compiled successfully.")

## Live Demo

**Scenario:** A research assistant is asked a multi-part question that requires a web search, a calculation, and a unit conversion — all potentially in parallel.

In [ ]:
question = (
    "What is the current population of Egypt? "
    "Also, if someone runs a marathon (42.195 km), how many miles is that? "
    "And what is 2 raised to the power of 16?"
)

print("Question:", question)
print("-" * 60)

result = graph.invoke({"messages": [HumanMessage(content=question)]})

In [ ]:
# Print the conversation trace
for msg in result["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n[{role}] Tool calls requested:")
        for tc in msg.tool_calls:
            print(f"  → {tc['name']}({tc['args']})")
    elif hasattr(msg, "name") and msg.name:  # ToolMessage
        print(f"\n[Tool:{msg.name}] Result: {msg.content[:200]}")
    else:
        print(f"\n[{role}] {msg.content}")

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Tool binding** | `llm.bind_tools(tools)` adds tool schemas to the LLM's context |
| **Parallel calls** | DeepSeek can request multiple tools in a single response |
| **Error isolation** | `ToolNode` wraps failures as `ToolMessage` — the graph keeps running |
| **`MessagesState`** | Built-in state type that appends rather than overwrites message history |

## What's Next

In **Notebook 03**, we build the full **ReAct** loop from scratch — no prebuilt `ToolNode`, no helper classes. Understanding the internals makes debugging and customization much easier.